# Curating EveBio IC50 activity data and pushing to a remote intake catalog

Mirrors the ChEMBL and BindingDB curation notebooks in this repo, but pulls from
[EveBio](https://evebio.org/)'s pharmome dataset (`eve-bio/drug-target-activity` on
Hugging Face) instead of a per-target API, since EveBio ships one bulk file covering all
their screened targets rather than a queryable endpoint.

EveBio's own schema doesn't label its measurement as "EC50" or "IC50" -- it reports a
fitted potency (`pXC50`) per assay `mode` (Agonist/Antagonist). Mechanistically, though,
these are exactly EC50 and IC50 in the standard sense: **Agonist**-mode TR-FRET
recruitment assays measure a concentration-response *increase* in signal (an EC50);
**Antagonist**-mode assays measure *inhibition* of a reference agonist's signal (an
IC50). This collection is named and organized accordingly
(`EveBio_EC50`/`EveBio_IC50`), matching `ChEMBL_pChEMBL_EC50`/`IC50` and
`BindingDB_EC50`/`IC50` elsewhere in this repo; `MODE` is kept internally only to filter
the EveBio source data by assay mode.

This dataset is gated and licensed CC BY-NC-SA 4.0 (non-commercial) -- different from the
permissive ChEMBL/BindingDB licenses already in this repo. See the collection README
before using this data commercially.

Targets: CAR (gene NR1I3, UniProt Q14994), FXR (gene NR1H4, UniProt Q96RI1), VDR (gene VDR, UniProt P11473), LXR_alpha (gene NR1H3, UniProt Q13133).


## Parameters

Everything that controls this run lives in this one cell. `PUSH_TO_S3` defaults to `False`.
`HF_TOKEN` must be set in the environment (an HF account that has accepted EveBio's gated
dataset terms); it is never read from or written into this notebook's source or output.

In [1]:
import os

TARGETS = {
    "CAR": ("NR1I3", "Q14994"),
    "FXR": ("NR1H4", "Q96RI1"),
    "VDR": ("VDR", "P11473"),
    "LXR_alpha": ("NR1H3", "Q13133"),
}  # name -> (gene symbol, UniProt accession)

MODE = 'Antagonist'  # EveBio's assay mode ("Agonist" or "Antagonist"); filters the source data
ENDPOINT_LABEL = 'IC50'  # "EC50" (Agonist) or "IC50" (Antagonist); see markdown above
SOURCE = "EveBio"
RETRIEVAL_DATE = '2026-10-08'
SOURCE_VERSION = RETRIEVAL_DATE.replace("-", "")  # EveBio's own `release` column varies
                                                    # per target/batch, not per whole-dataset
                                                    # snapshot, so retrieval date is used here,
                                                    # same convention as the BindingDB notebooks.
BUCKET = "openadmet-data-public-dev"
LOCATION = f"{SOURCE}{SOURCE_VERSION}_{ENDPOINT_LABEL}"
PUSH_TO_S3 = False  # set True only after reviewing local output; never push unreviewed data

HF_TOKEN = os.environ["HF_TOKEN"]  # must be set in the environment; raises KeyError otherwise


In [2]:
from pathlib import Path

import datamol as dm
import pandas as pd
import requests
from tqdm.auto import tqdm

from openadmet.toolkit.chemoinformatics.rdkit_funcs import canonical_smiles, smiles_to_inchikey

tqdm.pandas()

location_path = Path(LOCATION)
location_path.mkdir(exist_ok=True)


/Users/dwwest/miniconda3/envs/openadmet_toolkit/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Download the EveBio bulk file

EveBio ships one gated parquet file covering every target they've screened. We download
it to a scratch location (not committed -- it's a large, re-downloadable source file per
this repo's deposition guidelines) and filter to our targets and `MODE` locally.

In [3]:
import tempfile

def download_evebio_bulk():
    url = "https://huggingface.co/datasets/eve-bio/drug-target-activity/resolve/main/train.parquet"
    headers = {"Authorization": f"Bearer {HF_TOKEN}"}
    r = requests.get(url, headers=headers, timeout=300)
    r.raise_for_status()
    tmp_path = Path(tempfile.gettempdir()) / "evebio_train.parquet"
    tmp_path.write_bytes(r.content)
    return pd.read_parquet(tmp_path)

bulk = download_evebio_bulk()
print(f"downloaded {len(bulk)} total EveBio rows across all targets")


downloaded 737616 total EveBio rows across all targets


## Gather EveBio data for a target

EveBio's `target__uniprot_id` field is unreliable for heterodimerization-mode rows: it's
overwritten with the heterodimer partner's accession (`RXRA`) rather than the queried
target's own UniProt ID, and `target__chembl_target_id`/`target__pubchem_target_id` are
similarly unreliable there (e.g. `target__chembl_target_id` holding a UniProt accession,
`target__pubchem_target_id` holding a comma-joined mix of UniProt/ChEMBL/PubChem IDs for
the heterodimer pair). We match on `target__gene` instead, which stays correct across
assay mechanisms; `OPENADMET_TARGET_ID` is set from our own known-good UniProt mapping,
not copied from these source columns. Within `MODE`, rows from different assay mechanisms (co-factor
recruitment vs. heterodimerization, where both exist for a target) are pooled together in
the aggregate, same as how the ChEMBL/BindingDB permissive collections pool across assay
formats -- `assay__mechanism` is preserved per row in the raw file so this can be undone.

Only rows with a fitted `outcome_potency_pxc50` go into the aggregated file (EveBio's
equivalent of requiring a pChEMBL value); the much larger set of qualitative
active/inactive-only calls stays in the raw file but is excluded from the aggregate.

In [4]:
def gather_evebio_data_for_target(target_name: str, gene: str, uniprot_id: str, mode: str):
    print(f"working on target {target_name}")
    raw = bulk[(bulk["target__gene"] == gene) & (bulk["mode"] == mode)].copy()
    if raw.empty:
        return raw, raw

    print("canonicalising raw data")
    with dm.without_rdkit_log():
        raw["OPENADMET_CANONICAL_SMILES"] = raw["compound__smiles"].progress_apply(
            lambda x: canonical_smiles(x) if pd.notna(x) else pd.NA
        )
        raw["OPENADMET_INCHIKEY"] = raw["OPENADMET_CANONICAL_SMILES"].progress_apply(
            lambda x: smiles_to_inchikey(x) if pd.notna(x) else pd.NA
        )

    raw["OPENADMET_TARGET"] = target_name
    raw["OPENADMET_TARGET_ID"] = uniprot_id
    raw["OPENADMET_ENDPOINT"] = ENDPOINT_LABEL
    raw["OPENADMET_UNITS"] = "pXC50 (unitless, -log10 of unspecified XC50 in M)"
    raw["OPENADMET_SOURCE"] = SOURCE
    raw["OPENADMET_SOURCE_VERSION"] = SOURCE_VERSION

    raw = raw.rename(columns={"compound__inchikey": "compound_inchikey_source"})
    raw["release"] = raw["release"].astype("string")

    # all-null for every target in this deposition (EveBio's viability-counterscreen
    # linkage, unused by the NR agonist/antagonist assays pulled here)
    raw = raw.drop(columns=["viability_assay_id"])

    for col in ["assay_id", "target_id", "compound_id", "mode", "assay__mechanism",
                "assay__detailed_mechanism", "assay__technology", "frequency_flag",
                "target__class", "target__gene", "target__uniprot_id",
                "target__chembl_target_id", "target__pubchem_target_id",
                "target__wildtype_id", "target__name", "compound__name",
                "compound__smiles", "compound__drugbank_id", "compound__cas",
                "compound__unii", "compound_inchikey_source", "release",
                "pxc50_modifier",
                "OPENADMET_CANONICAL_SMILES", "OPENADMET_INCHIKEY", "OPENADMET_TARGET",
                "OPENADMET_TARGET_ID", "OPENADMET_ENDPOINT", "OPENADMET_UNITS",
                "OPENADMET_SOURCE", "OPENADMET_SOURCE_VERSION"]:
        raw[col] = raw[col].astype("string")

    g = raw.dropna(subset=["OPENADMET_CANONICAL_SMILES", "OPENADMET_INCHIKEY", "outcome_potency_pxc50"])
    aggregated = g.groupby(["OPENADMET_CANONICAL_SMILES", "OPENADMET_INCHIKEY"]).agg(
        pxc50_mean=("outcome_potency_pxc50", "mean"),
        pxc50_median=("outcome_potency_pxc50", "median"),
        pxc50_std=("outcome_potency_pxc50", "std"),
        n_records=("assay_id", "count"),
    ).reset_index()
    aggregated["n_records"] = aggregated["n_records"].astype("int64")
    aggregated["OPENADMET_TARGET"] = target_name
    aggregated["OPENADMET_TARGET_ID"] = uniprot_id
    aggregated["OPENADMET_ENDPOINT"] = ENDPOINT_LABEL
    aggregated["OPENADMET_UNITS"] = "pXC50 (unitless, -log10 of unspecified XC50 in M)"
    aggregated["OPENADMET_SOURCE"] = SOURCE
    aggregated["OPENADMET_SOURCE_VERSION"] = SOURCE_VERSION
    aggregated.sort_values("n_records", ascending=False, inplace=True)

    print("smiles duplicates", aggregated["OPENADMET_CANONICAL_SMILES"].duplicated().sum())
    print("inchikey duplicates", aggregated["OPENADMET_INCHIKEY"].duplicated().sum())
    print(f"  {len(raw)} raw rows ({raw['outcome_is_active'].sum()} active calls, "
          f"{raw['outcome_potency_pxc50'].notna().sum()} quantified), {len(aggregated)} aggregated compounds")
    return aggregated, raw


## Main loop

Generate the data for each target and save to local parquet. Pushing to S3 and (re)building
the Intake catalog only happens if `PUSH_TO_S3` is `True` -- see the Parameters cell.

In [5]:
uris_raw = {}
uris_agg = {}
for target, (gene, uniprot_id) in TARGETS.items():
    fname_agg = f"EveBio_{ENDPOINT_LABEL}_{target}_{uniprot_id}_aggregated.parquet"
    fname_raw = f"EveBio_{ENDPOINT_LABEL}_{target}_{uniprot_id}_raw.parquet"
    agg_path = location_path / fname_agg
    raw_path = location_path / fname_raw

    if agg_path.exists() and raw_path.exists():
        print(f"{target} already written locally, skipping")
        continue

    agg, raw = gather_evebio_data_for_target(target, gene, uniprot_id, MODE)
    if raw.empty:
        print(f"  no records for {target}, skipping")
        continue

    agg.reset_index(drop=True).to_parquet(agg_path, index=False)
    raw.reset_index(drop=True).to_parquet(raw_path, index=False)
    print(f"  wrote {fname_raw} (rows={len(raw)}), {fname_agg} (rows={len(agg)})")


working on target CAR
canonicalising raw data


  0%|          | 0/1397 [00:00<?, ?it/s]

  1%|          | 8/1397 [00:00<00:17, 77.34it/s]

  1%|          | 16/1397 [00:00<00:40, 34.28it/s]

  2%|▏         | 21/1397 [00:00<00:53, 25.62it/s]

...

 59%|█████▉    | 1645/2794 [00:00<00:00, 5625.00it/s]

 79%|███████▉  | 2215/2794 [00:00<00:00, 5651.51it/s]

100%|█████████▉| 2785/2794 [00:00<00:00, 5668.15it/s]

100%|██████████| 2794/2794 [00:00<00:00, 5551.24it/s]

smiles duplicates 0
inchikey duplicates 0
  2794 raw rows (224 active calls, 21 quantified), 16 aggregated compounds
  wrote EveBio_IC50_LXR_alpha_Q13133_raw.parquet (rows=2794), EveBio_IC50_LXR_alpha_Q13133_aggregated.parquet (rows=16)


## Push to S3 and build the Intake catalog

Guarded by `PUSH_TO_S3`. Reads AWS credentials from the environment (never shell out to
`aws configure get` in a committed notebook -- see the deposition guidelines, section 6).

In [6]:
if PUSH_TO_S3:
    import intake
    from intake.readers.datatypes import Parquet

    from openadmet.toolkit.webservices.credentials import S3Settings
    from openadmet.toolkit.webservices.s3 import S3Bucket

    assert "AWS_ACCESS_KEY_ID" in __import__("os").environ, "set AWS credentials in the environment first"
    settings = S3Settings()
    bucket = S3Bucket.from_settings(settings, BUCKET)

    cat = intake.entry.Catalog()
    for target, (gene, uniprot_id) in TARGETS.items():
        fname_agg = f"EveBio_{ENDPOINT_LABEL}_{target}_{uniprot_id}_aggregated.parquet"
        fname_raw = f"EveBio_{ENDPOINT_LABEL}_{target}_{uniprot_id}_raw.parquet"
        agg_path, raw_path = location_path / fname_agg, location_path / fname_raw
        if not (agg_path.exists() and raw_path.exists()):
            continue
        for path, fname, subset in [(agg_path, fname_agg, "aggregated"), (raw_path, fname_raw, "raw")]:
            dest = f"{LOCATION}/{fname}"
            bucket.push_file(path, dest)
            uri = bucket.to_uri(dest)
            row_desc = "one row per compound" if subset == "aggregated" else "one row per source record"
            metadata = {
                "description": f"{SOURCE}{SOURCE_VERSION} {ENDPOINT_LABEL} activity data for {target} ({uniprot_id}), {row_desc}.",
                "units": "pXC50 (unitless, -log10 of unspecified XC50 in M)",
                "endpoint": ENDPOINT_LABEL,
                "target": target,
                "target_uniprot_id": uniprot_id,
                "source_version": SOURCE_VERSION,
            }
            data = Parquet(url=uri, metadata=metadata)
            cat[f"{target}_{subset}"] = intake.readers.PandasParquet(data)

    catname = f"CATALOG_{LOCATION}.yaml"
    cat.to_yaml_file(catname)
    bucket.push_file(catname, f"{LOCATION}/{catname}")
    print(f"pushed catalog to {LOCATION}/{catname}")
else:
    print("PUSH_TO_S3 is False -- local parquet only, nothing pushed, no catalog built")


PUSH_TO_S3 is False -- local parquet only, nothing pushed, no catalog built
